# sanitise - Phase 3a backend on Kaggle

Same system as `backend.ipynb` (Colab), for Kaggle. Full steps: `docs/kaggle_setup.md`.

Before running: right sidebar -> **Session options** -> Accelerator **GPU T4 x2** (or P100),
**Internet On**, **Persistence: No persistence** is fine. **Add-ons -> Secrets**: add
`GEMINI_API_KEY`, `NGROK_AUTHTOKEN`, `NGROK_DOMAIN` and tick each as attached to this notebook.
Then run the cells in order.

| Cell | What it does |
|---|---|
| 1 Setup | Clone the repo, install packages, Ollama and llama3:8b, read the secrets |
| 2 Evaluate | Agents 1-3 on the 20 fixtures (optional) |
| 3 Restore | Bring back a store saved earlier (optional, instead of building) |
| 4 Build | Sanitise the corpus into the store and search index (resumes from checkpoints in this session) |
| 5 Save | Pack the store and index into `/kaggle/working/sanitise/store.tar.gz` |
| 6 Serve | Start the API and the tunnel; prints the address for the dashboard |
| 7 Smoke test | One question through the tunnel |
| 8 Target Info | Metric 1 |

Kaggle has no Google Drive. Files in `/kaggle/working` last only while the session lives; to keep
them, use **Save Version** (see the guide).


In [ ]:
# 1 Setup
# Edit these plain variables (Kaggle has no form fields).
MODEL = "llama3:8b"  # or "qwen2.5:7b-instruct"
# Agents 1-3: "gemini" = the Gemini API (free tier), key in the Kaggle secret GEMINI_API_KEY
# (Add-ons -> Secrets, attached to this notebook); several keys, comma-separated, are used in
# turn when one reaches its daily limit (each from a different Google Cloud project).
# "ollama" = a local model, AGENTS_MODEL below.
AGENTS_BACKEND = "gemini"  # or "ollama"
# GEMINI_MODEL may list several, comma-separated: the first is used, and the next takes over
# when it is overloaded (503) or out of daily requests.
GEMINI_MODEL = "gemini-3.5-flash-lite,gemini-3.1-flash-lite"
# For "ollama": Agents 2-3's model ("" = the same as MODEL) and its thinking switch.
AGENTS_MODEL = ""  # or "qwen3.5:4b", "qwen3.5:9b"
AGENTS_THINK = True
BRANCH = "main"  # main has everything, case files included

import os
import subprocess
import sys
import time
from pathlib import Path

WORK = Path("/kaggle/working")
os.environ["SANITISE_WORKDIR"] = str(WORK)  # tools/colab_cache.py keeps Ollama and its models here
REPO = WORK / "3rd_sem_el"
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH,
                    "https://github.com/raksha-sendhil/3rd_sem_el.git", str(REPO)], check=True)
os.chdir(REPO)
!git log --oneline -1

!pip install -q -r requirements.txt
!pip install -q -e ".[retrieval]" --no-deps
!pip install -q "sentence-transformers>=3.0" "chromadb>=0.5"
sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "tools"))
# Running this cell again after a pull must use the new code: forget modules already loaded.
for name in [m for m in sys.modules if m == "sanitise" or m.startswith(("sanitise.", "colab_cache"))]:
    del sys.modules[name]

import colab_cache

colab_cache.install_ollama_and_model(MODEL)
if AGENTS_BACKEND == "ollama" and AGENTS_MODEL:
    colab_cache.install_ollama_and_model(AGENTS_MODEL)

os.environ.update({
    "SANITISE_LLM_BACKEND": "ollama",
    "SANITISE_MODEL": MODEL,
    "SANITISE_LLM_TIMEOUT_S": "600",
    "SANITISE_RETRIEVAL_MODE": "hybrid",
})
for key in ("SANITISE_AGENTS_BACKEND", "SANITISE_AGENTS_MODEL", "SANITISE_AGENTS_THINK",
            "SANITISE_GEMINI_API_KEY"):
    os.environ.pop(key, None)
if AGENTS_BACKEND == "gemini":
    from kaggle_secrets import UserSecretsClient
    try:
        gemini_key = UserSecretsClient().get_secret("GEMINI_API_KEY")
    except Exception as exc:
        raise RuntimeError("Add the Kaggle secret GEMINI_API_KEY (Add-ons -> Secrets) and tick it "
                           "as attached to this notebook, then run this cell again.") from exc
    os.environ.update({"SANITISE_AGENTS_BACKEND": "gemini", "SANITISE_AGENTS_MODEL": GEMINI_MODEL,
                       "SANITISE_GEMINI_API_KEY": gemini_key})
    AGENTS = f"{GEMINI_MODEL} (Gemini API, {len(gemini_key.split(','))} key(s))"
elif AGENTS_MODEL:
    os.environ["SANITISE_AGENTS_MODEL"] = AGENTS_MODEL
    os.environ["SANITISE_AGENTS_THINK"] = "true" if AGENTS_THINK else "false"
    AGENTS = f"{AGENTS_MODEL} (thinking {'on' if AGENTS_THINK else 'off'})"
else:
    AGENTS = MODEL
AGENT1 = f"rules + {GEMINI_MODEL.split(',')[0]}" if AGENTS_BACKEND == "gemini" else f"rules + {MODEL}"
print("ready: Agent 1", AGENT1, "| Agents 2-3", AGENTS, "| RAG answers", MODEL)

In [ ]:
# 2 Evaluate Agents 1-3 on the 20 fixtures (copy the report at the end)

from sanitise.agents.pipeline import sanitise
from sanitise.config import get_settings
from sanitise.errors import PipelineError
from sanitise.llm.ollama import OllamaClient
from sanitise.loader import iter_documents

get_settings.cache_clear()
records, failures = [], {}
async with OllamaClient.from_settings(get_settings()) as llm:
    for doc in iter_documents("data/fixtures"):
        try:
            records.append(await sanitise(doc.doc_id, doc.text, llm))
        except PipelineError as exc:
            failures[doc.doc_id] = str(exc)
        print(".", end="", flush=True)
print()

out = WORK / "eval_fixtures"
out.mkdir(exist_ok=True)
for r in records:
    (out / f"{r.doc_id}.json").write_text(r.model_dump_json(indent=2))

lines = [f"AGENT 1 {AGENT1}  AGENTS 2-3 {AGENTS}  documents {len(records)}  failures {len(failures)}"]
lines.append(f"leak retries {sum(r.meta.leak_retries for r in records)}  "
             f"residual leaks {sum(len(r.residual_leaks) for r in records)}  "
             f"schema retries {sum(r.meta.retries for r in records)}  "
             f"masked words {sum(r.rewriter.safe_text.count('[REDACTED]') for r in records)}  "
             f"avg seconds {sum(r.meta.total_ms for r in records) / max(len(records), 1) / 1000:.1f}")
if Path("data/gold_facts").exists() and any(Path("data/gold_facts").glob("*.json")):
    from sanitise.evaluation.facts import score_facts
    from sanitise.schemas import ExtractorOutput, GoldRouting
    gold_facts = [ExtractorOutput.model_validate_json(p.read_text())
                  for p in sorted(Path("data/gold_facts").glob("*.json"))]
    lines.append(f"facts: {score_facts([r.extractor for r in records], gold_facts)}")
    if Path("data/gold_rewrite").exists() and any(Path("data/gold_rewrite").glob("*.json")):
        from sanitise.evaluation.rewrite import score_rewrite
        gold_routing = [GoldRouting.model_validate_json(p.read_text())
                        for p in sorted(Path("data/gold_rewrite").glob("*.json"))]
        lines.append(f"rewrite: {score_rewrite([r.rewriter for r in records], gold_routing, gold_facts)}")
for doc_id, err in failures.items():
    lines.append(f"FAILED {doc_id}: {err[:200]}")
for r in records:
    lines.append(f"--- {r.doc_id}  facts={len(r.extractor.facts)}  leak_retries={r.meta.leak_retries}")
    lines.append("FACTS: " + "; ".join(f"{f.subject}.{f.relation}={f.value}" for f in r.extractor.facts[:20]))
    lines.append("SAFE: " + r.rewriter.safe_text.replace(chr(10), " "))
report = "\n".join(lines)
(WORK / "eval_report.txt").write_text(report)
print(report)

In [ ]:
# 3 Restore a store saved earlier (optional; skip on the first run)
# Looks for store.tar.gz (from cell 5) or store_<N>.sqlite checkpoints in any attached input
# (a Dataset, or the output of an earlier version of this notebook) and under /kaggle/working.
import shutil

found_tar = sorted(Path("/kaggle/input").rglob("store.tar.gz"))
found_ckpt = sorted(Path("/kaggle/input").rglob("store_*.sqlite"))
if found_tar:
    subprocess.run(["tar", "-xzf", str(found_tar[0]), "-C", str(REPO)], check=True)
    print("restored store + index from", found_tar[0])
elif found_ckpt:
    (WORK / "sanitise").mkdir(exist_ok=True)
    for f in found_ckpt:
        shutil.copy(f, WORK / "sanitise" / f.name)
    print("copied checkpoints:", [f.name for f in found_ckpt], "- now run cell 4 with the same DOCS")
else:
    print("nothing found under /kaggle/input: run cell 4 to build")


In [ ]:
# 4 Build the store and search index from the corpus (safe to re-run: resumes from checkpoints)
# Sanitises a corpus with Agents 1-3, 25 documents at a time, and writes a checkpoint to
# /kaggle/working/sanitise/store_<DOCS>.sqlite after every step. About 16 s per document.
# DOCS: any number from 10 to 300, or "fixtures". Every size keeps the 3 attack targets.
# 20 = about 6 min (quick demo), 150 = about 40 min (the metric), 300 = about 1.5 h.
DOCS = "20"
SAVE = "/kaggle/working/sanitise"
FRESH = False  # True after a pipeline change: sanitise every document again

Path(SAVE).mkdir(parents=True, exist_ok=True)
FRESH_ARG = "--fresh" if FRESH else ""
!python tools/build_store.py --docs {DOCS} --checkpoint "{SAVE}/store_{DOCS}.sqlite" {FRESH_ARG}


In [ ]:
# 5 Save the store and index (tens of MB) so a later session can restore them
SAVED = WORK / "sanitise" / "store.tar.gz"
SAVED.parent.mkdir(parents=True, exist_ok=True)
!tar -czf "{SAVED}" data/store.sqlite data/index
print(f"saved {SAVED.stat().st_size / 1e6:.1f} MB to {SAVED}")
print("To keep it after this session ends: top right, Save Version (Save & Run All is not needed; "
      "Quick Save is enough), or download it from the Output tab.")


In [ ]:
# 6 Start the API and a public tunnel for the dashboard
# Tunnel: ngrok when its two Kaggle secrets are attached: NGROK_AUTHTOKEN (ngrok.com, free
# account -> Your Authtoken) and NGROK_DOMAIN (Domains -> your free static domain, e.g.
# name.ngrok-free.dev); the address then never changes. Otherwise, or if ngrok fails, a
# Cloudflare quick tunnel (no account, a new address each time, no uptime guarantee).
import re
import sys
import urllib.request


def api_health():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8000/v1/health", timeout=2).read().decode()
    except Exception:
        return None


colab_cache.start_ollama()
if api_health() is None:  # running this cell again keeps the API that is already up
    api_log = open(str(WORK / "api.log"), "a")
    api = subprocess.Popen(["sanitise", "serve", "--host", "0.0.0.0", "--port", "8000"],
                           stdout=api_log, stderr=subprocess.STDOUT, env=os.environ.copy())
    for _ in range(120):
        if api_health():
            break
        time.sleep(1)
print(api_health() or "API did not start: see /kaggle/working/api.log")


def kaggle_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return (UserSecretsClient().get_secret(name) or "").strip() or None
    except Exception:
        return None


def ngrok_tunnel():
    token = kaggle_secret("NGROK_AUTHTOKEN")
    if not token:
        print("ngrok: no NGROK_AUTHTOKEN secret, skipped")
        return None
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=True)
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = token
    ngrok.kill()
    domain = kaggle_secret("NGROK_DOMAIN")
    domain = re.sub(r"^https?://", "", domain).rstrip("/") if domain else None
    # newer ngrok agents take the fixed address as "url", older ones as "domain"
    attempts = [{"url": "https://" + domain}, {"domain": domain}] if domain else [{}]
    for options in attempts:
        try:
            return ngrok.connect(8000, "http", **options).public_url
        except Exception as exc:
            print("ngrok failed:", str(exc)[:300])
    return None


def cloudflare_tunnel(attempts=5):
    if not Path("/usr/local/bin/cloudflared").exists():
        subprocess.run(["wget", "-q", "-O", "/usr/local/bin/cloudflared",
                        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], check=True)
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)
    subprocess.run(["pkill", "-f", "cloudflared"])
    for attempt in range(1, attempts + 1):
        log = open(WORK / "tunnel.log", "w")
        tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8000", "--no-autoupdate"],
                                  stdout=log, stderr=subprocess.STDOUT)
        for _ in range(90):
            text = (WORK / "tunnel.log").read_text()
            found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", text)
            if found:
                return found[0]
            if "429" in text or tunnel.poll() is not None:
                break
            time.sleep(1)
        tunnel.kill()
        if attempt < attempts:
            print(f"cloudflare: attempt {attempt} failed, retrying in {30 * attempt} s")
            time.sleep(30 * attempt)
    print("cloudflare failed:", (WORK / "tunnel.log").read_text()[-800:])
    return None


URL = ngrok_tunnel() or cloudflare_tunnel()
if URL:
    print("API address for the dashboard:", URL)
    print("Set it in web/.env.local as NEXT_PUBLIC_API_URL=" + URL)
else:
    print("No tunnel. The API is still running; run this cell again in a few minutes.")

In [ ]:
# 7 Smoke test through the tunnel
import json as _json

HEADERS = {"content-type": "application/json", "ngrok-skip-browser-warning": "1"}

req = urllib.request.Request(
    URL + "/v1/sessions", data=b"{}", headers=HEADERS, method="POST")
session = _json.load(urllib.request.urlopen(req, timeout=60))["session_id"]
body = _json.dumps({"query": "Which patient has vertigo?", "session_id": session}).encode()
req = urllib.request.Request(URL + "/v1/compare", data=body,
                             headers=HEADERS, method="POST")
result = _json.load(urllib.request.urlopen(req, timeout=600))
print("BASELINE: ", result["baseline"]["answer"])
print("PROTECTED:", result["protected"]["answer"])

In [ ]:
# 8 Metric 1: Target Info (after cell 4 or 3; about 30-50 min for 150 documents; re-run resumes)
# Plays Person D's 251 adversarial queries once each against both pipelines and counts the true
# private values the answers reveal. Results go to /kaggle/working/sanitise/results/target_info/<RUN>/
# (report.md, report.json, answers.jsonl). Use Save Version afterwards to keep them.
RUN = ""  # empty: named after the store, ollama_<N>docs
SAVE = "/kaggle/working/sanitise"

if not Path("data/store.sqlite").exists():
    raise RuntimeError("No store on this runtime: run cell 4 (or 3) first.")
RESULTS = f"{SAVE}/results/target_info"
RUN_ARG = f'--run "{RUN}"' if RUN else ""
!python tools/target_info.py --out "{RESULTS}" {RUN_ARG}
latest = max(Path(RESULTS).glob("*/report.md"), key=lambda f: f.stat().st_mtime)
print(latest.read_text()[:4000])
